# Proyecto ICA 2026-I — Etapa 1: Planificacion
## LaLonde / NSW: Evaluacion de Metodos Observacionales mediante Benchmarking Experimental
**Estudiante:** Hugo Fernando Quiroz  
**Docente:** Dr. Jaime Lincovil Curivil  
**Propuesta:** Cat. 3, N.o 4 — Datasets Benchmark  
**Fecha:** 15 de mayo de 2026

---
### Estructura del notebook
```
00_setup/         <- instalacion de dependencias y semillas
01_dag/           <- construccion y validacion del DAG con pgmpy
02_datos/         <- carga, limpieza y EDA (Etapa 2)
03_estimacion/    <- PSM, AIPW, DML, SynC (Etapa 2-3)
04_sensibilidad/  <- placebo, E-value, Rosenbaum (Etapa 3)
05_resultados/    <- tablas y figuras finales (Etapa 3)
```

## 00. Setup — Dependencias y Semillas

In [ ]:
# === DEPENDENCIAS ===
# pip install dowhy econml causal-learn pgmpy pandas numpy scipy
#             scikit-learn matplotlib seaborn pot rpy2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Semilla global — DOCUMENTAR SIEMPRE
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Configuracion de visualizacion
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.family'] = 'sans-serif'
sns.set_theme(style='whitegrid')

print('Setup completo. RANDOM_STATE =', RANDOM_STATE)

## 01. DAG — Construccion y Validacion con pgmpy

In [ ]:
from pgmpy.models import BayesianNetwork
from pgmpy.independencies import Independencies

def construir_dag_nsw() -> BayesianNetwork:
    """
    Construye el DAG observado del estudio LaLonde/NSW.
    NOTA: La variable latente U no es modelable directamente en pgmpy;
    su efecto se analiza via analisis de sensibilidad en la Etapa 3.

    Aristas justificadas por:
    - LaLonde (1986): criterios de elegibilidad NSW
    - Dehejia & Wahba (1999): covariables de balance
    - Teoria economica del capital humano

    Returns
    -------
    BayesianNetwork: DAG del modelo causal
    """
    covars = ['age', 'educ', 'black', 'hispan', 'married', 'nodegree']
    proxies_U = ['re74', 're75']  # proxies parciales de U latente
    T, Y = 'treat', 're78'

    aristas = []
    # Confounders demograficos -> T
    aristas += [(c, T) for c in covars]
    # Confounders demograficos -> Y
    aristas += [(c, Y) for c in covars]
    # Proxies de U -> T y Y
    aristas += [(p, T) for p in proxies_U]
    aristas += [(p, Y) for p in proxies_U]
    # Efecto causal de interes
    aristas += [(T, Y)]

    dag = BayesianNetwork(aristas)
    return dag


dag = construir_dag_nsw()
print('Nodos:', dag.nodes())
print('Aristas:', dag.edges())
print('DAG valido (sin ciclos):', dag.check_model())

In [ ]:
def verificar_d_separacion(dag: BayesianNetwork,
                            X: str, Y: str,
                            condicionantes: list) -> bool:
    """
    Verifica si X e Y son d-separados dado el conjunto condicionante.

    Parameters
    ----------
    dag: BayesianNetwork
    X: str — variable de tratamiento
    Y: str — variable de resultado
    condicionantes: list — conjunto de ajuste

    Returns
    -------
    bool: True si d-separados (backdoor bloqueado)
    """
    return dag.local_independencies(X).contains(X, Y, condicionantes)


conjunto_ajuste = ['age', 'educ', 'black', 'hispan', 'married', 'nodegree', 're74', 're75']

# TODO (Etapa 1): verificar d-separacion con pgmpy
# Nota: U latente no esta en el DAG observado; la verificacion formal
# aplica al DAG con variables observadas unicamente.
print('Conjunto de ajuste:', conjunto_ajuste)
print('Criterio backdoor satisfecho bajo CIA: SI (justificado en seccion 3 del informe)')

## 02. Datos — Carga y EDA (Etapa 2)

In [ ]:
def cargar_datos_nsw(fuente: str = 'experimental') -> pd.DataFrame:
    """
    Carga el dataset LaLonde/NSW segun la fuente especificada.

    Parameters
    ----------
    fuente: str — 'experimental' | 'cps1' | 'psid'

    Returns
    -------
    pd.DataFrame con columnas estandarizadas
    """
    # TODO (Etapa 2): implementar descarga desde repositorio Dehejia
    # URL base: https://users.nber.org/~rdehejia/nswdata.html
    # Alternativa: pip install causalnlp
    raise NotImplementedError(f'Carga de datos {fuente} pendiente para Etapa 2')


def estadisticas_balance(df: pd.DataFrame,
                          covariables: list,
                          grupo: str = 'treat') -> pd.DataFrame:
    """
    Calcula estadisticas de balance estandarizadas (SMD) entre grupos.

    Parameters
    ----------
    df: pd.DataFrame
    covariables: list — lista de covariables a evaluar
    grupo: str — columna de tratamiento binario

    Returns
    -------
    pd.DataFrame con SMD por covariable
    """
    # TODO (Etapa 2): calcular Standardized Mean Difference (SMD)
    # SMD = (mu_1 - mu_0) / sqrt((sigma_1^2 + sigma_0^2) / 2)
    # Regla de decision: |SMD| < 0.1 indica balance adecuado
    pass


def plot_overlap(df: pd.DataFrame, ps_col: str = 'propensity_score') -> None:
    """
    Visualiza el overlap entre grupos mediante histogramas de propensity score.

    Parameters
    ----------
    df: pd.DataFrame
    ps_col: str — columna con propensity scores estimados
    """
    # TODO (Etapa 2): histograma superpuesto de PS por grupo tratamiento/control
    pass


print('Funciones de datos definidas. Implementacion en Etapa 2.')

## 03. Estimacion (Etapa 2-3)

In [ ]:
def estimar_psm(df: pd.DataFrame,
                outcome: str = 're78',
                tratamiento: str = 'treat',
                covariables: list = None,
                random_state: int = RANDOM_STATE) -> dict:
    """
    Estimacion ATT via Propensity Score Matching (PSM).
    Replica el metodo de Dehejia & Wahba (1999).

    Returns dict con: {'ATT': float, 'SE': float, 'IC_95': tuple}
    """
    # TODO (Etapa 2): implementar con sklearn LogisticRegression + matching 1:1
    pass


def estimar_aipw(df: pd.DataFrame,
                 outcome: str = 're78',
                 tratamiento: str = 'treat',
                 covariables: list = None,
                 random_state: int = RANDOM_STATE) -> dict:
    """
    Estimacion ATT via AIPW (Augmented IPW) — estimador doblemente robusto.
    Usa EconML o DoWhy para implementacion.

    ATT_AIPW = E[(mu_1(X) - mu_0(X)) * T / P(T=1)
               + T*(Y - mu_1(X)) / P(T=1)
               - (1-T)*P(T=1|X)*(Y - mu_0(X)) / (P(T=1)*P(T=0|X))] / P(T=1)

    Returns dict con: {'ATT': float, 'SE': float, 'IC_95': tuple}
    """
    # TODO (Etapa 2): implementar con econml.dr.LinearDRLearner
    pass


def estimar_dml(df: pd.DataFrame,
                outcome: str = 're78',
                tratamiento: str = 'treat',
                covariables: list = None,
                n_splits: int = 5,
                random_state: int = RANDOM_STATE) -> dict:
    """
    Estimacion ATT via Double Machine Learning con cross-fitting.
    Chernozhukov et al. (2018).

    Returns dict con: {'ATT': float, 'SE': float, 'IC_95': tuple}
    """
    # TODO (Etapa 2-3): implementar con econml.dml.DML
    # Modelos nuisance: RandomForest o GradientBoosting
    # Cross-fitting: KFold(n_splits=5, shuffle=True, random_state=random_state)
    pass


print('Funciones de estimacion definidas. Implementacion en Etapa 2-3.')

## 04. Sensibilidad (Etapa 3)

In [ ]:
def placebo_temporal(df: pd.DataFrame,
                     placebo_outcome: str = 're74',
                     tratamiento: str = 'treat',
                     covariables: list = None) -> dict:
    """
    Prueba placebo temporal: usa re74 (o re75) como variable de resultado.
    Si el estimador es valido, el efecto estimado deberia ser ~0
    (el tratamiento no puede afectar ingresos ANTERIORES a la intervencion).

    Returns dict con: {'ATT_placebo': float, 'p_valor': float}
    """
    # TODO (Etapa 3): reusar pipeline de estimacion con outcome = re74
    pass


def calcular_evalor(rr: float, rr_ud: float, rr_uy: float) -> float:
    """
    Calcula el E-value (VanderWeele & Ding, 2017).
    Mide cuan fuerte debe ser la confusion no observada para
    explicar el efecto estimado.

    E-value = RR + sqrt(RR * (RR - 1))

    Parameters
    ----------
    rr: float — Risk Ratio del efecto estimado
    rr_ud: float — RR entre U y D (tratamiento)
    rr_uy: float — RR entre U e Y (resultado)

    Returns
    -------
    float: E-value
    """
    return rr + np.sqrt(rr * (rr - 1))


def analisis_rosenbaum_gamma(ATT: float, SE: float,
                              gamma_range: list = None) -> pd.DataFrame:
    """
    Analisis de sensibilidad de Rosenbaum (2002).
    Evalua para que valor de Gamma (sesgo de seleccion) el
    intervalo de confianza incluye el cero.

    Parameters
    ----------
    ATT: float — efecto estimado
    SE: float — error estandar
    gamma_range: list — valores de Gamma a evaluar [1.0, 1.5, 2.0, 2.5, 3.0]

    Returns
    -------
    pd.DataFrame con p-valores de Rosenbaum por Gamma
    """
    # TODO (Etapa 3): implementar usando scipy.stats
    pass


print('Funciones de sensibilidad definidas. Implementacion en Etapa 3.')

## 05. DGP Sintetico — Validacion del Pipeline (Etapa 1)
*(Unico modulo completamente implementado en esta etapa)*

In [ ]:
def generar_dgp_nsw(n: int = 1000,
                    ATT_verdadero: float = 1794.0,
                    random_state: int = RANDOM_STATE) -> pd.DataFrame:
    """
    Genera datos sinteticos con estructura causal conocida, calibrada
    a las estadisticas descriptivas del NSW experimental.
    Permite calcular PEHE y verificar el pipeline antes de datos reales.

    Parameters
    ----------
    n: int — numero de observaciones
    ATT_verdadero: float — efecto causal verdadero (benchmark LaLonde 1986)
    random_state: int — semilla de reproducibilidad

    Returns
    -------
    pd.DataFrame con columnas: age, educ, black, hispan, married, nodegree,
                                re74, re75, treat, re78, tau (efecto individual)
    """
    rng = np.random.default_rng(random_state)

    # Covariables (calibradas a medias NSW experimental)
    age    = rng.normal(25.8, 7.2, n).clip(17, 55).astype(int)
    educ   = rng.normal(10.3, 2.0, n).clip(0, 16).astype(int)
    black  = (rng.uniform(0, 1, n) < 0.84).astype(int)
    hispan = (rng.uniform(0, 1, n) < 0.06).astype(int)
    married  = (rng.uniform(0, 1, n) < 0.19).astype(int)
    nodegree = (rng.uniform(0, 1, n) < 0.71).astype(int)
    re74   = np.maximum(rng.normal(2095, 5688, n), 0)
    re75   = np.maximum(rng.normal(1532, 3220, n), 0)

    # Propensity score (confusion observacional)
    logit = (-2.0 + 0.05*age - 0.1*educ + 0.3*black
             - 0.2*married + 0.1*nodegree
             - 0.0001*re74 - 0.0002*re75)
    ps = 1 / (1 + np.exp(-logit))
    treat = (rng.uniform(0, 1, n) < ps).astype(int)

    # Resultado potencial Y(0)
    Y0 = (500 + 80*educ - 100*nodegree + 0.3*re74 + 0.4*re75
          + rng.normal(0, 2000, n))
    Y0 = np.maximum(Y0, 0)

    # Efecto individual tau (heterogeneo)
    tau = ATT_verdadero + 200*(educ - 10) - 150*nodegree + rng.normal(0, 500, n)

    # Resultado observado
    re78 = Y0 + treat * tau

    df = pd.DataFrame({
        'age': age, 'educ': educ, 'black': black, 'hispan': hispan,
        'married': married, 'nodegree': nodegree,
        're74': re74.round(2), 're75': re75.round(2),
        'treat': treat, 're78': re78.round(2),
        'tau': tau.round(2)  # solo disponible en DGP sintetico
    })
    return df


# Generar y validar DGP
df_sint = generar_dgp_nsw(n=1000, ATT_verdadero=1794.0)

ATT_sint_verdadero = df_sint.loc[df_sint['treat']==1, 'tau'].mean()
ATT_sint_naive     = (df_sint.loc[df_sint['treat']==1, 're78'].mean()
                    - df_sint.loc[df_sint['treat']==0, 're78'].mean())

print(f'DGP sintetico generado: {len(df_sint)} obs, {df_sint["treat"].sum()} tratados')
print(f'ATT verdadero (tau medio sobre tratados): ${ATT_sint_verdadero:,.0f}')
print(f'Estimador naive (diff de medias sin ajuste): ${ATT_sint_naive:,.0f}')
print(f'Sesgo de seleccion: ${ATT_sint_naive - ATT_sint_verdadero:,.0f}')
print()
print(df_sint.describe().round(1))

---
## Proximos pasos (Etapa 2)
1. Descargar datos reales NSW (experimental + CPS-1 + PSID)
2. Implementar `estadisticas_balance()` y `plot_overlap()`
3. Implementar y correr `estimar_psm()` — benchmarking Dehejia & Wahba
4. Implementar `estimar_aipw()` con EconML
5. Primer E-value de sensibilidad preliminar

**Entrega E2:** viernes 26 de junio de 2026, 23:59 (hora Lima)